# Getting Started with BaseTable

## Introduction

This tutorial demonstrates how to use ``BaseTableSchema``, ``TableManifestation``, and ``Database`` to define tables and perform CRUD operations using **sqlalchemyobjects**.

The Unified Workflow involves:
1. Defining a **TableSchema Mixin** (the blueprint logic).
2. Defining a **TableManifestation** (the interface).
3. Defining the **Database Schema** (SQLAlchemy Base + Tables).
4. Defining the **Database** class (the orchestrator).

This tutorial will guide you through:
- Defining schema mixins and manifestations
- Setting up the database and schema
- Performing CRUD operations (Create, Read, Update, Delete)
- Handling Asynchronous operations

**Prerequisites:**
- Basic familiarity with Python and SQLAlchemy
- Installed package: ``sqlalchemyobjects``

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Examples](#Examples)
- [Async Usage](#Async-Usage)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

We start by importing the necessary classes from ``sqlalchemyobjects`` and standard SQLAlchemy components.

In [ ]:
from pathlib import Path
from typing import Any
from sqlalchemy.orm import Mapped, mapped_column, Session, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseTableSchema, TableManifestation

## Core Functionality

The core workflow separates logic (Mixin), interface (Manifestation), and implementation (Schema/DB).

### 1. Define the Schema Mixin

The TableSchema defines the columns and core logic. It inherits from ``BaseTableSchema`` but NOT ``DeclarativeBase``. This allows it to be reused.

In [ ]:
class UserTableSchema(BaseTableSchema):
    """A mixin defining the user table structure and logic."""
    name: Mapped[str]
    role: Mapped[str] = mapped_column(default="user")

    @classmethod
    def get_by_name(cls, session: Session, name: str) -> Any:
        # Custom query method at the schema level
        return cls.get_by(session, "name", name).scalars().first()

### 2. Define the Manifestation

The manifestation is the interface to your table. It wraps the schema logic and manages sessions.

In [ ]:
class UserTableManifestation(TableManifestation):
    """A session-aware interface for the User table."""
    def get_by_name(self, name: str, session: Session | None = None) -> Any:
        if session is None:
            with self.create_session() as session:
                return self.table_schema.get_by_name(session, name)
        else:
            return self.table_schema.get_by_name(session, name)

### 3. Define the Database Schema

Define the SQLAlchemy ``DeclarativeBase`` and create the final table classes by combining the TableSchema with the Base.

In [ ]:
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base class for the database schema."""

class UserTable(UserTableSchema, DatabaseSchema):
    """SQLAlchemy table definition for Users, combining schema and base."""
    __tablename__ = "users"

### 4. Define the Database Class

Subclass ``Database`` to tie everything together. Map the manifestation to the table using ``table_map``.

In [ ]:
class MyDatabase(Database):
    """Database class managing the User table."""
    schema = DatabaseSchema
    table_map = {
        "users": (UserTableManifestation, UserTable, {})
    }

    @property
    def users(self) -> UserTableManifestation:
        return self.tables["users"]

### 5. Initialize the Database

Instantiate your database class.

In [ ]:
db_path = Path("tutorial_db.sqlite")
if db_path.exists():
    db_path.unlink()

# Initialize Database
database = MyDatabase(path=db_path)
database.create_database()

# Access the table via the property
users = database.users

## Examples

Now we can use the ``users`` manifestation object to interact with our database.

In [ ]:
# Create
print("Inserting users...")
users.insert({"name": "Alice", "role": "admin"})
users.insert({"name": "Bob"})

# Read
print("Querying users...")
alice = users.get_by_name("Alice")
if alice:
    print(f"Found: {alice.name} ({alice.role})")

# Update (Upsert)
print("Updating Bob...")
bob = users.get_by_name("Bob")
# Upsert allows updating by primary key (ID)
users.upsert({"id": bob.id, "role": "developer"})

bob_updated = users.get_by_name("Bob")
print(f"Bob's new role: {bob_updated.role}")

# Delete
print("Deleting Alice...")
users.delete(alice.id)
count = users.count()
print(f"Remaining users: {count}")

## Async Usage

``sqlalchemyobjects`` supports asynchronous operations. You just need to instantiate your Database with ``async_engine=True``.

In [ ]:
import anyio

async def async_demo():
    async_db_path = anyio.Path("tutorial_async.sqlite")
    if await async_db_path.exists():
        await async_db_path.unlink()

    # Initialize with async_engine=True using the SAME MyDatabase class
    async_db = MyDatabase(path=str(async_db_path), async_engine=True)
    await async_db.create_database_async()

    # Access the table
    users_async = async_db.users

    # Insert Async
    await users_async.insert_async({"name": "Charlie", "role": "async-user"})

    # Query Async
    # Note: Custom async queries would need async implementation in Schema/Manifestation.
    # Here we use the built-in generic count_async
    count = await users_async.count_async()
    print(f"Async Count: {count}")

    await async_db.close_async()
    await async_db_path.unlink()

# Run the async function
# Note: In a real script, use asyncio.run(async_demo())
await async_demo()

In [ ]:
# Cleanup synchronous database
database.close()
if db_path.exists():
    db_path.unlink()

## API Highlights

- **``BaseTableSchema``**: Inherited by TableSchemas (logic).
- **``TableManifestation``**: Inherited by Manifestations (interface).
- **``Database``**: Inherited by your Database class (orchestrator).
  - **``table_map``**: Maps names to (Manifestation, TableClass, kwargs).

## Troubleshooting / FAQs

- **Problem**: `AttributeError: 'NoneType' object has no attribute '...'`
  - **Solution**: Ensure your query actually returned an object. ``get_by_name`` returns ``None`` if no user is found.

- **Problem**: Database tables are not created.
  - **Solution**: Ensure ``database.create_database()`` is called after defining the schema.

## Conclusion and Next Steps

You have learned the standard workflow for defining tables and performing operations with ``sqlalchemyobjects``.

- **Next**: Explore ``SingletonTableManifestation`` for configuration tables.
- **Reference**: See ``docs/concepts/comprehensive.rst`` for more details on the Unified Workflow.